In [ ]:
import os
import math

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn

from datasets import load_from_disk, concatenate_datasets
from brainlm_mae.modeling_brainlm import BrainLMForPretraining

In [ ]:
if not os.path.exists("inference_plots"):
    os.mkdir("inference_plots")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Load model

In [ ]:
model = BrainLMForPretraining.from_pretrained("pretrained_models/2023-06-24-00_00_00_checkpoint-3000")
model

In [ ]:
print(model.vit.config)

In [ ]:
print(model.vit.embeddings.mask_ratio)
print(model.vit.embeddings.config.mask_ratio)

In [ ]:
model.vit.embeddings.mask_ratio = 0.0
model.vit.embeddings.config.mask_ratio = 0.0

## Load Entire Dataset

In [ ]:
train_ds = load_from_disk("/home/sr2464/palmer_scratch/datasets/UKBioBank1000_Arrow_v4/train_ukbiobank1000")
print(train_ds)
val_ds = load_from_disk("/home/sr2464/palmer_scratch/datasets/UKBioBank1000_Arrow_v4/val_ukbiobank1000")
print(val_ds)
test_ds = load_from_disk("/home/sr2464/palmer_scratch/datasets/UKBioBank1000_Arrow_v4/test_ukbiobank1000")
print(test_ds)
coords_ds = load_from_disk("/home/sr2464/palmer_scratch/datasets/UKBioBank1000_Arrow_v4/Brain_Region_Coordinates")
print(coords_ds)

In [ ]:
concat_ds = concatenate_datasets([train_ds, val_ds, test_ds])
concat_ds

In [ ]:
example0 = concat_ds[0]
print(example0['Filename'])
print(example0['Patient ID'])
print(example0['Order'])
print(example0['eid'])
print(example0['Gender'])
print(example0['Age.At.MHQ'])
print(example0['Depressed.At.Baseline'])
print(example0['Neuroticism'])
print(example0['Self.Harm.Ever'])
print(example0['Not.Worth.Living'])
print(example0['PCL.Score'])
print(example0['GAD7.Severity'])

## Forward Pass Through Model, Pass Whole fMRI Recording

In [ ]:
variable_of_interest_col_name = "Age.At.MHQ"
recording_col_name = "Subtract_Mean_Divide_Global_STD_Normalized_Recording"

def preprocess_fmri(examples):
    """
    Preprocessing function for dataset samples. This function is passed into Trainer as
    a preprocessor which takes in one row of the loaded dataset and constructs a model
    input sample according to the arguments which model.forward() expects.

    The reason this function is defined inside on main() function is because we need
    access to arguments such as cell_expression_vector_col_name.
    """
    label = examples[variable_of_interest_col_name][0]
    if math.isnan(label):
        label = -1  # replace nans with -1
    else:
        label = int(label)
    label = torch.tensor(label, dtype=torch.int64)
    signal_vector = examples[recording_col_name][0]
    signal_vector = torch.tensor(signal_vector, dtype=torch.float32)

    # Choose random starting index, take window of moving_window_len points for each region
    start_idx = 0
    end_idx = 490  # 24 patches per voxel, * 424 = 10176 total per sample
    signal_window = signal_vector[start_idx: end_idx, :]  # [moving_window_len, num_voxels]
    signal_window = torch.movedim(signal_window, 0, 1)  # --> [num_voxels, moving_window_len]

    # Append signal values and coords
    window_xyz_list = []
    for brain_region_idx in range(signal_window.shape[0]):
        # window_timepoint_list = torch.arange(0.0, 1.0, 1.0 / num_timepoints_per_voxel)

        # Append voxel coordinates
        xyz = torch.tensor([
            coords_ds[brain_region_idx]["X"],
            coords_ds[brain_region_idx]["Y"],
            coords_ds[brain_region_idx]["Z"]
        ], dtype=torch.float32)
        window_xyz_list.append(xyz)
    window_xyz_list = torch.stack(window_xyz_list)

    # Add in key-value pairs for model inputs which CellLM is expecting in forward() function:
    #  signal_vectors and xyz_vectors
    #  These lists will be stacked into torch Tensors by collate() function (defined above).
    examples["signal_vectors"] = signal_window.unsqueeze(0)
    examples["xyz_vectors"] = window_xyz_list.unsqueeze(0)
    examples["label"] = label
    return examples

In [ ]:
def collate_fn(example):
    """
    This function tells the dataloader how to stack a batch of examples from the dataset.
    Need to stack gene expression vectors and maintain same argument names for model inputs
    which CellLM is expecting in forward() function:
        expression_vectors, sampled_gene_indices, and cell_indices
    """
    # These inputs will go to model.forward(), names must match
    return {
        "signal_vectors": example["signal_vectors"],
        "xyz_vectors": example["xyz_vectors"],
        "input_ids": example["signal_vectors"],
        "labels": example["label"]
    }

In [ ]:
#--- Forward 1 sample through just the model encoder (model.vit) ---#
with torch.no_grad():
    example1 = concat_ds[0]
    
    # Wrap each value in the key:value pairs into a list (expected by preprocess() and collate())
    example1[recording_col_name] = [example1[recording_col_name]]
    example1[variable_of_interest_col_name] = [example1[variable_of_interest_col_name]]

    processed_example1 = preprocess_fmri(example1)
    encoder_output = model.vit(
        signal_vectors=processed_example1["signal_vectors"],
        xyz_vectors=processed_example1["xyz_vectors"],
        output_attentions=True,
        output_hidden_states=True
    )

In [ ]:
print("last_hidden_state:", encoder_output.last_hidden_state.shape)
# [batch_size, num_genes + 1 CLS token, hidden_dim]

cls_token = encoder_output.last_hidden_state[:,0,:]
print(cls_token.shape)

## Do For All Recordings, Save CLS Tokens

In [ ]:
all_cls_tokens = []
with torch.no_grad():
    for recording_idx in tqdm(range(concat_ds.num_rows)):
        example1 = concat_ds[recording_idx]

        # Wrap each value in the key:value pairs into a list (expected by preprocess() and collate())
        example1[recording_col_name] = [example1[recording_col_name]]
        example1[variable_of_interest_col_name] = [example1[variable_of_interest_col_name]]

        processed_example1 = preprocess_fmri(example1)
        encoder_output = model.vit(
            signal_vectors=processed_example1["signal_vectors"],
            xyz_vectors=processed_example1["xyz_vectors"],
            output_attentions=True,
            output_hidden_states=True
        )

        cls_token = encoder_output.last_hidden_state[:,0,:]  # torch.Size([1, 256])
        all_cls_tokens.append(cls_token.detach().cpu().numpy())

In [ ]:
all_cls_tokens = np.concatenate(all_cls_tokens, axis=0)

In [ ]:
all_cls_tokens.shape

In [ ]:
np.save("inference_plots/all_cls_tokens_{}recordinglength.npy".format(490), all_cls_tokens)

In [ ]:
# Save raw recordings as well
all_recordings = []
for recording_idx in tqdm(range(concat_ds.num_rows)):
    example1 = concat_ds[recording_idx]
    recording = np.array(example1[recording_col_name], dtype=np.float32)
    recording = recording[:490].flatten()
    all_recordings.append(recording)

all_recordings = np.stack(all_recordings, axis=0)
all_recordings.shape

In [ ]:
np.save("inference_plots/all_{}_490len.npy".format(recording_col_name), all_recordings)

In [ ]:
# Reload CLS tokens and raw data if needed
all_cls_tokens = np.load("inference_plots/2023-06-24-00_00_00_checkpoint-3000/all_cls_tokens_490recordinglength.npy")
all_cls_tokens.shape

In [ ]:
all_recordings = np.load("inference_plots/all_{}.npy".format(recording_col_name))

In [ ]:
all_recordings.shape

# Plotting

Visualize CLS tokens visualized by different variables of interest (Age, Neurocity, etc). Make it a reusable function

In [ ]:
import umap
import matplotlib.pyplot as plt
import matplotlib.colors as mcol
import seaborn as sns

In [ ]:
# sns.set_style()
sns.reset_orig()

In [ ]:
def umap_visualize_CLS_tokens_no_labels(cls_tokens):
    # Apply UMAP
    reducer = umap.UMAP(random_state=42)
    embedding = reducer.fit_transform(cls_tokens)

    # Plot
    plt.figure(figsize=(6, 6))
    plt.rcParams.update({'font.size': 18})
    sc = plt.scatter(embedding[:, 0].tolist(), embedding[:, 1].tolist(), alpha=0.7)
    plt.title("CLS Token UMAP", fontsize=16)
    plt.xlabel("UMAP Dimension 1")
    plt.ylabel("UMAP Dimension 2")
    plt.savefig("inference_plots/umap_cls_tokens_uncolored.png", 
                facecolor="white", bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
umap_visualize_CLS_tokens_no_labels(all_cls_tokens)

In [ ]:
def umap_visualize_raw_data_no_labels(raw_data):
    # Apply UMAP
    reducer = umap.UMAP(random_state=42)
    embedding = reducer.fit_transform(raw_data)

    # Plot
    plt.figure(figsize=(6, 6))
    plt.rcParams.update({'font.size': 18})
    sc = plt.scatter(embedding[:, 0].tolist(), embedding[:, 1].tolist(), alpha=0.7)
    plt.title("Raw Recordings UMAP\n({})".format(recording_col_name), fontsize=16)
    plt.xlabel("UMAP Dimension 1")
    plt.ylabel("UMAP Dimension 2")
    plt.savefig("inference_plots/umap_raw_data_uncolored.png", 
                facecolor="white", bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
umap_visualize_raw_data_no_labels(raw_data=all_recordings)

In [ ]:
concat_ds

In [ ]:
concat_ds.features.keys()

In [ ]:
print(concat_ds["Gender"][:5])
print(concat_ds["Age.At.MHQ"][:5])
print(concat_ds["PHQ9.Severity"][:5])
print(concat_ds["Depressed.At.Baseline"][:5])
print(concat_ds["Neuroticism"][:5])
print(concat_ds["Self.Harm.Ever"][:5])
print(concat_ds["Not.Worth.Living"][:5])
print(concat_ds["PCL.Score"][:5])
print(concat_ds["GAD7.Severity"][:5])

In [ ]:
def umap_visualize_CLS_tokens(cls_tokens, concat_ds, variable="Age.At.MHQ"):
    assert variable in concat_ds.features.keys(), \
    "Please specify a column in the dataset for variable of interest"
    variable_list = concat_ds[variable]
    var_name = variable.replace(".", "_")
    
    # Apply UMAP
    reducer = umap.UMAP(random_state=42)
    embedding = reducer.fit_transform(cls_tokens)
    num_missing = sum([1 if math.isnan(num) else 0 for num in variable_list])

    # Plot
    plt.figure(figsize=(8, 6))
    plt.rcParams.update({'font.size': 18})
    sc = plt.scatter(embedding[:, 0].tolist(), embedding[:, 1].tolist(),
                     alpha=0.7, c=variable_list, plotnonfinite=False, cmap="viridis")  # , cmap=colormap
    plt.title("CLS Token UMAP Colored By {}\n({}/{} samples with missing\nvalues omitted)"
              .format(variable, num_missing, len(variable_list)), 
              fontsize=16)
    plt.xlabel("UMAP Dimension 1")
    plt.ylabel("UMAP Dimension 2")
    plt.colorbar(sc, label=variable)
    plt.savefig("inference_plots/umap_cls_tokens_{}_colored.png".format(var_name), 
                facecolor="white", bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
for variable_name in ['Gender', 'Age.At.MHQ', 'PHQ9.Severity', 'Depressed.At.Baseline', 'Neuroticism', 
                      'Self.Harm.Ever', 'Not.Worth.Living', 'PCL.Score', 'GAD7.Severity']:
    umap_visualize_CLS_tokens(all_cls_tokens, concat_ds, variable=variable_name)

### Visualize raw data through UMAP

In [ ]:
def umap_visualize_raw_data(raw_data, concat_ds, variable="Age.At.MHQ"):
    assert variable in concat_ds.features.keys(), \
    "Please specify a column in the dataset for variable of interest"
    variable_list = concat_ds[variable]
    var_name = variable.replace(".", "_")
    
    # Apply UMAP
    reducer = umap.UMAP(random_state=42)
    embedding = reducer.fit_transform(raw_data)
    num_missing = sum([1 if math.isnan(num) else 0 for num in variable_list])

    # Plot
    plt.figure(figsize=(8, 6))
    plt.rcParams.update({'font.size': 18})
    sc = plt.scatter(embedding[:, 0].tolist(), embedding[:, 1].tolist(),
                     alpha=0.7, c=variable_list, plotnonfinite=False, cmap="viridis")  # , cmap=colormap
    plt.title("Raw Data UMAP Colored By {}\n({}/{} samples with missing\nvalues omitted)"
              .format(variable, num_missing, len(variable_list)), 
              fontsize=16)
    plt.xlabel("UMAP Dimension 1")
    plt.ylabel("UMAP Dimension 2")
    plt.colorbar(sc, label=variable)
    plt.savefig("inference_plots/umap_raw_data_{}_colored.png".format(var_name), 
                facecolor="white", bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
for variable_name in ['Gender', 'Age.At.MHQ', 'PHQ9.Severity', 'Depressed.At.Baseline', 'Neuroticism', 
                      'Self.Harm.Ever', 'Not.Worth.Living', 'PCL.Score', 'GAD7.Severity']:
    umap_visualize_raw_data(all_recordings, concat_ds, variable=variable_name)

# Perform PCA on CLS Tokens and Raw Recording

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
print(all_cls_tokens.shape)
print(all_recordings.shape)

In [ ]:
n_components = 200
pca = PCA(n_components=n_components)

In [ ]:
pca.fit(all_cls_tokens)

In [ ]:
print("pca.n_components_:", pca.n_components_)
print("pca.n_features_in_:", pca.n_features_in_)

In [ ]:
print("pca.components_.shape:", pca.components_.shape)
print("pca.explained_variance_.shape:", pca.explained_variance_.shape)
print("pca.explained_variance_ratio_.shape:", pca.explained_variance_ratio_.shape)
print("pca.singular_values_.shape:", pca.singular_values_.shape)

In [ ]:
print(pca.explained_variance_[:10])
print(pca.explained_variance_ratio_[:10])
print(pca.explained_variance_ratio_.sum())

In [ ]:
plt.figure(figsize=(8, 4))
pc_index = np.arange(50) + 1
plt.bar(pc_index, pca.explained_variance_ratio_[:50], label="Explained Variance Per Component")
plt.plot(pc_index, pca.explained_variance_ratio_[:50].cumsum(), 'o-', 
         linewidth=2, color='red', label="Cumulative Variance Explained")
plt.title('Explained Variance for Principal Components of\nReduced CLS Tokens', fontsize=14)
plt.xlabel('Principal Component', fontsize=14)
plt.ylabel('% Variance Explained', fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(fontsize=14)
plt.savefig("inference_plots/pca_scree_plot_cls_tokens_50components.png", bbox_inches="tight", facecolor="white")
plt.show()
plt.close()

In [ ]:
cls_tokens_pca_reduced = pca.transform(all_cls_tokens)
cls_tokens_pca_reduced.shape

In [ ]:
np.save("inference_plots/pca_reduced_cls_tokens_200components.png", cls_tokens_pca_reduced)

Reduce raw data

In [ ]:
n_components = 200
pca = PCA(n_components=n_components)

In [ ]:
pca.fit(all_recordings)

In [ ]:
print("pca.n_components_:", pca.n_components_)
print("pca.n_features_in_:", pca.n_features_in_)

In [ ]:
print("pca.components_.shape:", pca.components_.shape)
print("pca.explained_variance_.shape:", pca.explained_variance_.shape)
print("pca.explained_variance_ratio_.shape:", pca.explained_variance_ratio_.shape)
print("pca.singular_values_.shape:", pca.singular_values_.shape)

In [ ]:
print(pca.explained_variance_[:10])
print(pca.explained_variance_ratio_[:10])
print(pca.explained_variance_ratio_.sum())

In [ ]:
plt.figure(figsize=(8, 4))
pc_index = np.arange(200) + 1
plt.bar(pc_index, pca.explained_variance_ratio_, label="Explained Variance Per Component")
plt.plot(pc_index, pca.explained_variance_ratio_.cumsum(), 'o-', markersize=6,
         linewidth=2, color='red', label="Cumulative Variance Explained")
plt.title('Explained Variance for Principal Components of\nRaw Recordings', fontsize=14)
plt.xlabel('Principal Component', fontsize=14)
plt.ylabel('% Variance Explained', fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(fontsize=14)
plt.savefig("inference_plots/pca_scree_plot_raw_data_490len_200components.png", bbox_inches="tight", facecolor="white")
plt.show()
plt.close()

In [ ]:
raw_data_pca_reduced = pca.transform(all_recordings)
print(raw_data_pca_reduced.shape)
print(raw_data_pca_reduced[:, :50].shape)

In [ ]:
np.save("inference_plots/pca_reduced_raw_data_490len_200components.png", raw_data_pca_reduced)

In [ ]:
raw_data_pca_reduced_10components = raw_data_pca_reduced[:, :10]
cls_tokens_pca_reduced_10_components = cls_tokens_pca_reduced[:, :10]

In [ ]:
print(raw_data_pca_reduced_10components.shape)
print(cls_tokens_pca_reduced_10_components.shape)

# Plotting CLS Tokens and Raw Recordings With PCA

In [ ]:
def pca_visualize_CLS_tokens(cls_token_pca_reduction, concat_ds, variable="Age.At.MHQ"):
    assert variable in concat_ds.features.keys(), \
    "Please specify a column in the dataset for variable of interest"
    variable_list = concat_ds[variable]
    var_name = variable.replace(".", "_")
    num_missing = sum([1 if math.isnan(num) else 0 for num in variable_list])

    # Plot
    plt.figure(figsize=(8, 6))
    plt.rcParams.update({'font.size': 18})
    sc = plt.scatter(cls_token_pca_reduction[:, 0].tolist(), cls_token_pca_reduction[:, 1].tolist(),
                     alpha=0.7, c=variable_list, plotnonfinite=False, cmap='viridis')
    plt.title("CLS Token First Two Principal Components\nColored By {} ({}/{} samples\nwith missing values omitted)"
              .format(variable, num_missing, len(variable_list)), 
              fontsize=16)
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.colorbar(sc, label=variable)
    plt.savefig("inference_plots/pca_cls_tokens_{}_colored.png".format(var_name), 
                facecolor="white", bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
for variable_name in ['Gender', 'Age.At.MHQ', 'PHQ9.Severity', 'Depressed.At.Baseline', 'Neuroticism', 
                      'Self.Harm.Ever', 'Not.Worth.Living', 'PCL.Score', 'GAD7.Severity']:
    pca_visualize_CLS_tokens(cls_tokens_pca_reduced_10_components, concat_ds, variable=variable_name)

In [ ]:
def pca_visualize_raw_data(raw_data_pca_reduction, concat_ds, variable="Age.At.MHQ"):
    assert variable in concat_ds.features.keys(), \
    "Please specify a column in the dataset for variable of interest"
    variable_list = concat_ds[variable]
    var_name = variable.replace(".", "_")
    num_missing = sum([1 if math.isnan(num) else 0 for num in variable_list])

    # Plot
    plt.figure(figsize=(8, 6))
    plt.rcParams.update({'font.size': 18})
    sc = plt.scatter(raw_data_pca_reduction[:, 0].tolist(), raw_data_pca_reduction[:, 1].tolist(),
                     alpha=0.7, c=variable_list, plotnonfinite=False, cmap='viridis')
    plt.title("Raw Recording First Two Principal Components\nColored By {} ({}/{} samples\nwith missing values omitted)"
              .format(variable, num_missing, len(variable_list)), 
              fontsize=16)
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.colorbar(sc, label=variable)
    plt.savefig("inference_plots/pca_raw_data_{}_colored.png".format(var_name), 
                facecolor="white", bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
for variable_name in ['Gender', 'Age.At.MHQ', 'PHQ9.Severity', 'Depressed.At.Baseline', 'Neuroticism', 
                      'Self.Harm.Ever', 'Not.Worth.Living', 'PCL.Score', 'GAD7.Severity']:
    pca_visualize_raw_data(raw_data_pca_reduced_10components, concat_ds, variable=variable_name)